# 02 — Stage B: Data, Split, Exploration and Features

**Steps 3–7 of the 11-step predictive-modeling process.**

Stage A (`01_problem_framing.ipynb`) settled what is being predicted, what errors cost,
and which metric decides success. This notebook turns the raw file into a modelling table
without ever letting the test period influence a decision.

Sections are numbered by **process step**, but they run in dependency order. The one
place the two differ is Step 5:

| Order | Step | Section | What happens | Leakage status |
|---|---|---|---|---|
| 1st | **3** | 3.1–3.7 | Data dictionary, granularity, load, dates, target | safe — fixed rules |
| 2nd | **4** | 4 | **Out-of-time split at `issue_d = 2016-01-01`** | **everything after respects it** |
| 3rd | **6** | 6.1–6.4 | Logic checks, cleaning, missingness, column screening | screening is train-only |
| 4th | **7** | 7 | Feature engineering (row-wise only) | safe on full frame |
| 5th | **5** | 5 | EDA on **pre-cutoff rows only** | choices must not see test |

**Why Step 5 runs last.** EDA cannot precede cleaning here: in the raw file `int_rate` is
the string `"13.56%"` and `term` is `" 36 months"`, so there is nothing to plot until
Step 6 has parsed them, and several figures use features built in Step 7. The process
itself notes it is *a loop, not a line*, and this is that loop — what matters is that EDA
happens **before any modelling decision**, which it does, and that it sees **training rows
only**, which it also does.

**Why Step 4 runs second, before all of them.** The process is explicit that the split is
decided *before* exploring. An analyst who has studied the test period has already leaked
it into the model through their own choices, and no metric will ever reveal it.

Imputation, scaling and encoding are deliberately **not** applied here. They learn
parameters from data, so they live inside the `Pipeline` in notebook 03 and are fitted on
training rows only. See `CLAUDE.md` for the full rules.

> **Note on the discussion cells.** Each figure is followed by a caption, a discussion and
> an explicit **Action →** line, since the process requires every chart to lead to a
> concrete action. Those discussions describe the patterns expected on the **real Kaggle
> data**. If you are running on the synthetic stand-in, re-read them against your actual
> output and revise before submitting.

## Step 3 — Collect and understand the data

### 3.1 Setup

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Reproducibility -------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# The frozen out-of-time cutoff. Shared with every other group on this dataset.
# Changing it means regenerating artifacts/split_manifest.json and telling them.
CUTOFF_DATE = pd.Timestamp("2016-01-01")

# --- Paths -----------------------------------------------------------------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
ARTIFACTS = ROOT / "artifacts"
FIG_DIR = ARTIFACTS / "figures"
for _d in (DATA_RAW, DATA_PROCESSED, ARTIFACTS, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

RAW_FILE = DATA_RAW / "accepted_2007_to_2018Q4.csv.gz"
SYNTHETIC_FILE = DATA_RAW / "synthetic_sample.parquet"

# --- Plot styling ----------------------------------------------------------------
# Validated categorical palette (see CLAUDE.md / dataviz palette reference).
C_BLUE, C_ORANGE, C_AQUA, C_VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
C_RED, C_GRAY = "#e34948", "#8a8a86"
OUTCOME_COLORS = {0: C_BLUE, 1: C_RED}   # 0 = repaid, 1 = charged off
OUTCOME_LABELS = {0: "Fully Paid", 1: "Charged Off"}

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
})


def save_fig(name: str) -> None:
    """Save the current figure to artifacts/figures/<name>.png and display it."""
    plt.savefig(FIG_DIR / f"{name}.png")
    plt.show()


print(f"Project root : {ROOT}")
print(f"OOT cutoff   : {CUTOFF_DATE.date()}")

### 3.2 Column specification, granularity and the timing test

Two explicit lists drive the whole notebook.

`APPLICATION_COLUMNS` — the fields a lender genuinely has **at the moment of scoring**.
Restricting `read_csv` to these keeps memory in check on the 1.6 GB file and makes the
information set an explicit, reviewable decision rather than an accident.

`LEAKAGE_COLUMNS` — fields recorded **after** the funding decision. Every one of them is
a partial function of the outcome we are trying to predict, so any model that sees them
scores near-perfectly in validation and is worthless in deployment. They are excluded
from `usecols` and dropped again defensively in step 6.

Free-text fields (`emp_title`, `title`) are excluded: tens of thousands of unique values,
no ordinal structure, and a natural vehicle for accidental target encoding.

### 3.3 Granularity, provenance and the timing test

**Granularity: one row = one _funded_ loan application.**

Three consequences follow, and each one changes a later decision:

1. **Not one row per borrower.** A borrower who takes two loans appears twice. The process
   flags this as the `GroupKFold` case — the same entity landing in both train and test.
   Here it is handled by the out-of-time split instead: the same borrower can only span
   the boundary by applying in two different periods, and those are genuinely different
   applications with different credit files. The dataset also carries no stable borrower
   key, so grouping is not available even if we wanted it.
2. **Not one row per _application_.** Rejected applications are absent entirely — they
   live in a separate Lending Club file with no outcome recorded. This is the source of
   the selection bias documented in notebook 03.
3. **Not a time series per entity.** Each row resolves once. The *dataset* has a time
   dimension (`issue_d`) which drives the split, but each row is an independent
   observation, not a sequence.

**Provenance.** Lending Club's public loan book, 2007–2018 Q4, as published on Kaggle
(`wordsforthewise/lending-club`). Self-reported applicant fields (income, employment,
purpose) are **unverified** unless `verification_status` says otherwise — treat them as
claims, not facts.

**The timing test.** For every column the process asks: *"on the day the prediction is
needed, does this column already have a value?"* The data dictionary below answers it
column by column. Any "no" is a leakage column and is excluded.

In [ ]:
# The data dictionary: meaning, unit, and -- the column that matters -- whether the
# value exists at scoring time. This is the machine-readable answer to the timing test.
DATA_DICTIONARY = [
    # (column, meaning, unit, known at application?)
    ("loan_amnt",        "Amount requested by the applicant",        "USD",      True),
    ("term",             "Repayment period",                         "months",   True),
    ("int_rate",         "Interest rate set by Lending Club",        "%",        True),
    ("installment",      "Scheduled monthly payment",                "USD",      True),
    ("grade",            "LC risk grade A-G",                        "ordinal",  True),
    ("sub_grade",        "LC risk sub-grade A1-G5",                  "ordinal",  True),
    ("purpose",          "Applicant-stated use of funds (unverified)", "category", True),
    ("initial_list_status", "Listing type at origination",           "category", True),
    ("application_type", "Individual or joint application",          "category", True),
    ("emp_length",       "Years in current employment",              "years",    True),
    ("home_ownership",   "Housing status (self-reported)",           "category", True),
    ("annual_inc",       "Self-reported annual income",              "USD",      True),
    ("verification_status", "Whether LC verified the income",        "category", True),
    ("addr_state",       "US state of residence",                    "category", True),
    ("dti",              "Debt-to-income, excluding this loan",      "%",        True),
    ("delinq_2yrs",      "Delinquencies in the last 2 years",        "count",    True),
    ("earliest_cr_line", "Date the oldest credit line was opened",   "date",     True),
    ("fico_range_low",   "Lower bound of FICO band at application",  "score",    True),
    ("fico_range_high",  "Upper bound of FICO band at application",  "score",    True),
    ("inq_last_6mths",   "Credit inquiries in the last 6 months",    "count",    True),
    ("mths_since_last_delinq", "Months since last delinquency",      "months",   True),
    ("open_acc",         "Open credit lines",                        "count",    True),
    ("pub_rec",          "Derogatory public records",                "count",    True),
    ("revol_bal",        "Revolving balance",                        "USD",      True),
    ("revol_util",       "Revolving line utilisation",               "%",        True),
    ("total_acc",        "Total credit lines ever",                  "count",    True),
    ("mort_acc",         "Mortgage accounts",                        "count",    True),
    ("pub_rec_bankruptcies", "Public-record bankruptcies",           "count",    True),
    ("issue_d",          "Month the loan was issued (time-proxy)",   "date",     True),
    ("loan_status",      "Outcome -- becomes the TARGET",            "category", False),
    # --- Known only AFTER funding: the leakage block ---
    ("recoveries",       "Post-charge-off recovery amount",          "USD",      False),
    ("total_pymnt",      "Total repaid to date",                     "USD",      False),
    ("total_rec_prncp",  "Principal repaid to date",                 "USD",      False),
    ("last_pymnt_d",     "Date of most recent payment",              "date",     False),
    ("out_prncp",        "Outstanding principal",                    "USD",      False),
    ("last_fico_range_low", "FICO re-pulled DURING the loan",        "score",    False),
    ("debt_settlement_flag", "Loan entered settlement",              "flag",     False),
    ("hardship_flag",    "Borrower entered hardship plan",           "flag",     False),
    ("funded_amnt",      "Amount actually funded (post-decision)",   "USD",      False),
]

data_dict = pd.DataFrame(DATA_DICTIONARY,
                         columns=["column", "meaning", "unit", "known_at_application"])
data_dict["usable_as_feature"] = data_dict["known_at_application"]
data_dict.loc[data_dict["column"] == "loan_status", "usable_as_feature"] = False  # it IS the target

print(f"{data_dict['known_at_application'].sum()} columns pass the timing test; "
      f"{(~data_dict['known_at_application']).sum()} fail it and are excluded.")
display(data_dict.set_index("column").style.set_caption(
    "Table 3 - Data dictionary. The 'known_at_application' column is the timing test."))

**Table 3 — Data dictionary with the timing test applied.**

The `known_at_application` column is the whole point. Every `False` is a field that exists
only because the loan was already funded and its outcome partly known — `recoveries` is
nonzero only after a charge-off, `total_pymnt` is smaller for loans that defaulted,
`last_fico_range_*` moves *with* the default event itself.

A model given any of them scores near-perfectly in validation and fails completely in
production, because at scoring time those columns are all empty. This is the indirect
leakage the process warns is far more dangerous than forgetting to drop the target.

**Action →** `APPLICATION_COLUMNS` (the `True` rows) drives `usecols` at load time, and
`LEAKAGE_COLUMNS` (the `False` rows) is re-applied defensively in 3.7. Any future column
added to the project must be added to this table first, with its timing answered.

In [ ]:
# Fields available at application time.
APPLICATION_COLUMNS = [
    # Loan request
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "purpose", "initial_list_status", "application_type",
    # Applicant
    "emp_length", "home_ownership", "annual_inc", "verification_status", "addr_state",
    # Credit file at application
    "dti", "delinq_2yrs", "earliest_cr_line", "fico_range_low", "fico_range_high",
    "inq_last_6mths", "mths_since_last_delinq", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc", "mort_acc", "pub_rec_bankruptcies",
    # Timing and outcome
    "issue_d", "loan_status",
]

# Post-origination fields. NEVER use these as features.
LEAKAGE_COLUMNS = [
    "recoveries", "collection_recovery_fee",              # only nonzero after charge-off
    "total_pymnt", "total_pymnt_inv",                     # cumulative repayment
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",    # payment history
    "out_prncp", "out_prncp_inv",                         # outstanding principal
    "last_fico_range_low", "last_fico_range_high",        # FICO re-pulled during the loan
    "last_credit_pull_d",
    "debt_settlement_flag", "debt_settlement_flag_date",  # distressed loans only
    "settlement_status", "settlement_date", "settlement_amount",
    "settlement_percentage", "settlement_term",
    "hardship_flag", "hardship_type", "hardship_status", "hardship_amount",
    "funded_amnt", "funded_amnt_inv",                     # post-decision; use loan_amnt
]

print(f"{len(APPLICATION_COLUMNS)} application-time columns")
print(f"{len(LEAKAGE_COLUMNS)} leakage columns blocked")

### 3.4 Load the data

If `data/raw/accepted_2007_to_2018Q4.csv.gz` is present it is used. If not, the notebook
generates a **synthetic sample** with the same column names, dtypes and plausible value
ranges — including the messy string encodings of the real file (`" 36 months"`,
`"13.56%"`, `"10+ years"`) — so the cleaning code downstream is genuinely exercised.

The synthetic frame spans 2012–2018 with a mild upward drift in the default rate, so the
out-of-time split and the drift discussion are non-degenerate.

**Synthetic results are for smoke-testing the pipeline only.** Never quote them as
findings. See the README for how to download the real file.

In [ ]:
def make_synthetic_sample(n: int = 40_000, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Generate a stand-in for the Kaggle file with the same schema and messy encodings."""
    rng = np.random.default_rng(seed)

    # --- Issue dates: 2012-2018, with growing origination volume ------------------
    months = pd.date_range("2012-01-01", "2018-12-01", freq="MS")
    vol_weights = np.linspace(0.4, 2.2, len(months))
    vol_weights /= vol_weights.sum()
    issue_d = pd.to_datetime(rng.choice(months.values, size=n, p=vol_weights))
    t = (issue_d - months[0]).days / 365.25          # years since start, for drift

    # --- Grade drives price and risk ---------------------------------------------
    grades = np.array(list("ABCDEFG"))
    grade_p = np.array([0.19, 0.29, 0.27, 0.15, 0.07, 0.024, 0.006])
    g_idx = rng.choice(len(grades), size=n, p=grade_p)
    grade = grades[g_idx]
    sub_grade = np.char.add(grade, rng.integers(1, 6, size=n).astype(str))
    int_rate = np.clip(6.0 + 2.9 * g_idx + rng.normal(0, 1.1, n), 5.3, 30.9)

    # --- Applicant finances -------------------------------------------------------
    annual_inc = np.clip(rng.lognormal(11.0, 0.55, n), 4_000, 1_500_000)
    loan_amnt = np.clip(
        np.round((annual_inc * rng.uniform(0.05, 0.45, n)) / 25) * 25, 500, 40_000)
    term_months = rng.choice([36, 60], size=n, p=[0.72, 0.28])

    # Standard amortisation, so `installment` is internally consistent.
    r = int_rate / 100 / 12
    installment = np.round(loan_amnt * r / (1 - (1 + r) ** -term_months), 2)

    # --- Credit file --------------------------------------------------------------
    fico_low = np.clip(
        np.round((740 - 11 * g_idx + rng.normal(0, 22, n)) / 5) * 5, 660, 845)
    dti = np.clip(rng.gamma(4.0, 4.6, n) + 0.9 * g_idx, 0, 60)
    revol_util = np.clip(rng.beta(2.6, 2.6, n) * 100 + 2.2 * g_idx, 0, 145)
    open_acc = np.clip(rng.poisson(11, n), 1, 70)
    total_acc = open_acc + np.clip(rng.poisson(14, n), 0, 90)
    revol_bal = np.round(np.clip(rng.lognormal(9.2, 1.0, n), 0, 900_000))
    delinq_2yrs = rng.poisson(0.22 + 0.05 * g_idx, n)
    inq_last_6mths = rng.poisson(0.55 + 0.09 * g_idx, n)
    pub_rec = rng.poisson(0.13, n)
    pub_rec_bankruptcies = np.minimum(pub_rec, rng.poisson(0.09, n))
    mort_acc = rng.poisson(1.5, n)
    credit_hist_years = np.clip(rng.gamma(6.0, 2.6, n), 1, 60)
    earliest_cr_line = issue_d - pd.to_timedelta(credit_hist_years * 365.25, unit="D")

    emp_length = rng.choice(
        ["< 1 year", "1 year", "2 years", "3 years", "4 years", "5 years", "6 years",
         "7 years", "8 years", "9 years", "10+ years"],
        size=n,
        p=[0.08, 0.07, 0.10, 0.09, 0.07, 0.07, 0.05, 0.05, 0.04, 0.04, 0.34])
    home_ownership = rng.choice(["MORTGAGE", "RENT", "OWN", "OTHER"],
                                size=n, p=[0.49, 0.40, 0.105, 0.005])
    verification_status = rng.choice(
        ["Verified", "Source Verified", "Not Verified"], size=n, p=[0.30, 0.36, 0.34])
    purpose = rng.choice(
        ["debt_consolidation", "credit_card", "home_improvement", "other",
         "major_purchase", "medical", "small_business", "car"],
        size=n, p=[0.55, 0.22, 0.07, 0.06, 0.035, 0.025, 0.02, 0.02])
    addr_state = rng.choice(
        ["CA", "NY", "TX", "FL", "IL", "NJ", "PA", "OH", "GA", "VA", "NC", "MI"],
        size=n, p=[0.16, 0.10, 0.09, 0.08, 0.06, 0.06, 0.06, 0.05, 0.05, 0.05, 0.05, 0.19])

    # --- Latent default risk ------------------------------------------------------
    # Signal deliberately modest so metrics land near the honest 0.68-0.72 AUC band.
    logit = (-2.55
             + 0.235 * g_idx
             + 0.020 * (dti - 18)
             + 2.05 * (loan_amnt / annual_inc - 0.20)
             - 0.0072 * (fico_low - 700)
             + 0.0056 * (revol_util - 50)
             + 0.085 * inq_last_6mths
             + 0.115 * delinq_2yrs
             + 0.33 * (term_months == 60)
             + 0.055 * t                      # upward drift over the window
             + rng.normal(0, 0.65, n))
    default_prob = 1 / (1 + np.exp(-logit))
    defaulted = rng.random(n) < default_prob

    loan_status = np.where(defaulted, "Charged Off", "Fully Paid").astype(object)
    # Loans issued late in the window are more likely to be unmatured -> "Current".
    still_current = rng.random(n) < np.clip((t - 4.2) / 3.6, 0, 0.62)
    loan_status[still_current] = "Current"
    loan_status[rng.random(n) < 0.006] = "Late (31-120 days)"

    df = pd.DataFrame({
        "loan_amnt": loan_amnt,
        "term": [f" {m} months" for m in term_months],           # messy, like the real file
        "int_rate": [f"{v:.2f}%" for v in int_rate],             # messy
        "installment": installment,
        "grade": grade,
        "sub_grade": sub_grade,
        "purpose": purpose,
        "initial_list_status": rng.choice(["w", "f"], size=n, p=[0.62, 0.38]),
        "application_type": rng.choice(["Individual", "Joint App"], size=n, p=[0.95, 0.05]),
        "emp_length": emp_length,
        "home_ownership": home_ownership,
        "annual_inc": annual_inc,
        "verification_status": verification_status,
        "addr_state": addr_state,
        "dti": dti,
        "delinq_2yrs": delinq_2yrs,
        "earliest_cr_line": earliest_cr_line.strftime("%b-%Y"),  # messy, like the real file
        "fico_range_low": fico_low,
        "fico_range_high": fico_low + 4,
        "inq_last_6mths": inq_last_6mths,
        "mths_since_last_delinq": rng.integers(1, 120, n).astype(float),
        "open_acc": open_acc,
        "pub_rec": pub_rec,
        "revol_bal": revol_bal,
        "revol_util": [f"{v:.1f}%" for v in revol_util],         # messy
        "total_acc": total_acc,
        "mort_acc": mort_acc,
        "pub_rec_bankruptcies": pub_rec_bankruptcies,
        "issue_d": issue_d.strftime("%b-%Y"),                    # messy, like the real file
        "loan_status": loan_status,
    })

    # Realistic missingness. mths_since_last_delinq is missing STRUCTURALLY -- it is blank
    # precisely for applicants who have never been delinquent, so the blank carries signal
    # and the Step 6.3 missing-indicator has something real to find.
    never_delinquent = (delinq_2yrs == 0) & (rng.random(n) < 0.85)
    df.loc[never_delinquent, "mths_since_last_delinq"] = np.nan
    df.loc[rng.random(n) < 0.05, "emp_length"] = np.nan
    df.loc[rng.random(n) < 0.004, "revol_util"] = np.nan
    df.loc[rng.random(n) < 0.02, "dti"] = np.nan

    # A handful of deliberately corrupt rows, so the Step 6.1 data-logic checks have
    # something to catch and the drop path is exercised by the smoke test.
    corrupt = rng.choice(n, size=max(6, n // 4000), replace=False)
    half = len(corrupt) // 2
    df.loc[corrupt[:half], "annual_inc"] = 0                 # impossible income
    df.loc[corrupt[half:], "fico_range_high"] = 600          # inverted FICO band
    return df

In [ ]:
def load_raw() -> tuple[pd.DataFrame, str]:
    """Return (frame, source_label). Falls back to the synthetic sample when needed."""
    wanted = set(APPLICATION_COLUMNS)

    if RAW_FILE.exists():
        print(f"Reading real data: {RAW_FILE}")
        frame = pd.read_csv(RAW_FILE, usecols=lambda c: c in wanted, low_memory=False)
        return frame, "kaggle"

    print(f"!! {RAW_FILE.name} not found -- falling back to the SYNTHETIC sample.")
    print("!! Results are for smoke-testing only. See README.md to download the real file.")
    if SYNTHETIC_FILE.exists():
        return pd.read_parquet(SYNTHETIC_FILE), "synthetic"

    frame = make_synthetic_sample()
    frame.to_parquet(SYNTHETIC_FILE, index=False)
    print(f"   generated and cached -> {SYNTHETIC_FILE}")
    return frame, "synthetic"


df, DATA_SOURCE = load_raw()
print(f"\nLoaded {len(df):,} rows x {df.shape[1]} columns  (source: {DATA_SOURCE})")
df.head()

### 3.5 Parse dates

Row-wise and deterministic — each row's output depends only on that row — so this is safe
to run on the full frame before splitting. It has to happen first, because the split
itself is defined on `issue_d`.

The real file stores these as `"Dec-2015"` strings.

In [ ]:
for col in ("issue_d", "earliest_cr_line"):
    if not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], format="%b-%Y", errors="coerce")

n_before = len(df)
df = df.dropna(subset=["issue_d"]).copy()
print(f"Dropped {n_before - len(df):,} rows with an unparseable issue_d")
print(f"Date range: {df['issue_d'].min():%Y-%m} to {df['issue_d'].max():%Y-%m}")

### 3.6 Define the target variable

A credit-scoring label needs a **resolved** outcome. Only terminal statuses qualify:

| `loan_status` | Target | Reason |
|---|---|---|
| `Fully Paid` | `0` | Resolved, repaid |
| `Charged Off`, `Default` | `1` | Resolved, defaulted |
| `Current`, `Late …`, `In Grace Period`, `Issued` | *dropped* | Outcome not yet known |

Dropping unresolved loans is necessary but not free — it is the source of the
right-censoring bias recorded in the limitations of notebook 03.

This is a fixed rule with no learned parameter, so it is safe on the full frame.

In [ ]:
POSITIVE_STATUSES = {"Charged Off", "Default"}
NEGATIVE_STATUSES = {"Fully Paid"}

status_counts = df["loan_status"].value_counts()
print("Raw loan_status distribution:")
print(status_counts.to_string(), "\n")

terminal = df["loan_status"].isin(POSITIVE_STATUSES | NEGATIVE_STATUSES)
print(f"Terminal (usable)   : {terminal.sum():>9,}")
print(f"Unresolved (dropped): {(~terminal).sum():>9,}")

df = df.loc[terminal].copy()
df["target"] = df["loan_status"].isin(POSITIVE_STATUSES).astype(int)
df = df.drop(columns=["loan_status"])

default_rate = df["target"].mean()
print(f"\nRows kept: {len(df):,}   default rate: {default_rate:.2%}"
      f"   (imbalance ratio 1:{(1 - default_rate) / default_rate:.1f})")

### 3.7 Defensive leakage-column drop

`usecols` already excluded these, so this normally removes nothing. It stays as a guard:
if someone later widens `APPLICATION_COLUMNS`, this cell catches the mistake instead of
letting a 0.99 AUC quietly look like success.

In [ ]:
present = [c for c in LEAKAGE_COLUMNS if c in df.columns]
if present:
    print(f"WARNING -- dropping {len(present)} leakage columns that slipped in: {present}")
    df = df.drop(columns=present)
else:
    print("OK -- no post-origination leakage columns present.")

## Step 4 — Split strategy (decided BEFORE exploring)

### 4.1 The shared out-of-time split

> **Protocol.** Sort by `issue_d`. Loans issued **before 2016-01-01** are training data;
> loans issued **on or after 2016-01-01** are test data.

**Why out-of-time rather than random.** Default prediction is a forecasting problem.
Applicants arrive in time order, and across 2007–2018 Lending Club's product mix, credit
policy and macroeconomic backdrop all drift. A random split lets the model train on 2018
loans in order to predict 2015 ones — information a deployed scorecard could never have.
The result is an optimistic estimate of a model that will be used on *next* quarter's
applicants, not on a random sample of the past.

**Why a date is a better shared protocol than a seed.** The brief requires every group on
this dataset to use the same split. A frozen date depends on no library RNG, so it
reproduces exactly across implementations and library versions — `train_test_split`'s
shuffle does not.

The random stratified split is computed here **as a reference only**, so notebook 03 can
report how much optimism random splitting buys. It is never the headline result.

In [ ]:
from sklearn.model_selection import train_test_split

# Chronological order; ties broken deterministically so the manifest hash is stable.
df = df.sort_values(["issue_d", "loan_amnt", "int_rate"], kind="mergesort").reset_index(drop=True)

# --- Primary protocol: out-of-time -------------------------------------------------
df["is_train"] = df["issue_d"] < CUTOFF_DATE

# --- Reference only: random stratified ---------------------------------------------
ref_train_idx, ref_test_idx = train_test_split(
    df.index.to_numpy(), test_size=0.2, random_state=RANDOM_SEED, stratify=df["target"])
df["is_train_random_ref"] = df.index.isin(ref_train_idx)

train_mask, test_mask = df["is_train"], ~df["is_train"]
summary = pd.DataFrame({
    "rows": [train_mask.sum(), test_mask.sum()],
    "share": [train_mask.mean(), test_mask.mean()],
    "default_rate": [df.loc[train_mask, "target"].mean(), df.loc[test_mask, "target"].mean()],
    "first_issue": [df.loc[train_mask, "issue_d"].min(), df.loc[test_mask, "issue_d"].min()],
    "last_issue": [df.loc[train_mask, "issue_d"].max(), df.loc[test_mask, "issue_d"].max()],
}, index=["train (pre-cutoff)", "test (post-cutoff)"])
display(summary)

# Hard guard against look-ahead.
assert df.loc[train_mask, "issue_d"].max() < df.loc[test_mask, "issue_d"].min(), \
    "Look-ahead detected: train and test issue_d ranges overlap."
assert train_mask.sum() > 0 and test_mask.sum() > 0, "One side of the split is empty."
print("OK -- no temporal overlap between train and test.")

The manifest recording this split is written at the **end** of this notebook, not
here. Steps 6.1 and 6.2 still remove rows (logic violations, duplicates), and the manifest
must describe the frame that is actually exported — otherwise its row counts disagree with
`clean.parquet` and the assertion in notebook 03 fails. The cutoff rule itself is fixed
here and never changes; only the counts are settled later.

## Step 6 — Clean the data

### 6.1 Data-logic checks

Before imputing anything, check the data says something **possible**. The process calls
for exactly this — negative ages, sale dates before build dates — and the credit-file
equivalents are below. Each check is a statement about the world that the data must
satisfy; a violation is either a parsing bug on our side or a data-quality defect on
Lending Club's.

Violations are reported with counts first and only then acted on, because the count is
itself diagnostic: a handful of rows is a data defect, while thousands usually means our
parsing is wrong.

In [ ]:
# Each check: (name, boolean mask of VIOLATING rows, what to do about it)
checks = {
    "earliest_cr_line after issue_d (credit file opened after the loan)":
        df["earliest_cr_line"] > df["issue_d"],
    "fico_range_high below fico_range_low (inverted FICO band)":
        df["fico_range_high"] < df["fico_range_low"],
    "annual_inc at or below zero":
        df["annual_inc"] <= 0,
    "dti negative":
        df["dti"] < 0,
    "loan_amnt at or below zero":
        df["loan_amnt"] <= 0,
    "open_acc exceeds total_acc (more open than total lines)":
        df["open_acc"] > df["total_acc"],
}

logic_report = pd.DataFrame(
    [{"check": name, "violations": int(mask.sum()), "share": float(mask.mean())}
     for name, mask in checks.items()]).set_index("check")
display(logic_report.style.format({"violations": "{:,}", "share": "{:.4%}"})
        .set_caption("Table 4 - Data-logic checks. Every row here should be near zero."))

# Rows violating a hard impossibility are dropped: they cannot be repaired without
# inventing data, and imputing over them would hide a real data-quality problem.
impossible = (checks["earliest_cr_line after issue_d (credit file opened after the loan)"]
              | checks["fico_range_high below fico_range_low (inverted FICO band)"]
              | checks["annual_inc at or below zero"]
              | checks["loan_amnt at or below zero"])
n_before = len(df)
df = df.loc[~impossible].reset_index(drop=True)
print(f"\nDropped {n_before - len(df):,} rows violating a hard constraint "
      f"({(n_before - len(df)) / max(n_before, 1):.3%} of the data)")
print(f"Remaining: {len(df):,} rows")

**Table 4 — Data-logic checks.**

All counts should be at or near zero. Rows breaking a hard impossibility are dropped
rather than repaired: there is no honest way to invent a credit-history start date, and
imputing over the problem would conceal a data-quality signal we would rather see.

`open_acc > total_acc` is reported but **not** dropped — it is implausible rather than
impossible, and on the real file it reflects reporting lag between bureau fields.

**Action →** If any count here is large on the real data, stop and re-check the parsing in
3.5 before continuing; a mass violation almost always means a date format or unit was
misread, not that Lending Club published thousands of impossible loans.

### 6.1b Normalise category labels

The process warns that `"HN"`, `"Hà Nội"` and `"ha noi"` must become one label. The Lending
Club equivalent is `home_ownership`, which carries `ANY`, `NONE` and `OTHER` — three
labels for what is operationally the same residual category, an artefact of the form
changing over the years. Left alone they become three sparse dummies that mean the same
thing, and worse, their *relative frequencies drift over time*, so the encoder would learn
a different vocabulary for the training period than the test period contains.

In [ ]:
HOME_OWNERSHIP_NORMALISATION = {"ANY": "OTHER", "NONE": "OTHER"}

before = df["home_ownership"].value_counts()
df["home_ownership"] = df["home_ownership"].replace(HOME_OWNERSHIP_NORMALISATION)
after = df["home_ownership"].value_counts()

comparison = pd.concat([before.rename("before"), after.rename("after")], axis=1).fillna(0)
display(comparison.astype(int).style.set_caption(
    "Table 5 - home_ownership label normalisation (ANY/NONE folded into OTHER)"))

**Table 5 — Category normalisation.**

**Action →** Any new categorical column must get the same treatment before it reaches the
one-hot encoder in notebook 03. Check `value_counts()` for near-synonyms whenever a column
is added.

### 6.2 Row-wise cleaning

Everything here is deterministic and per-row, so it is safe on the full frame:

- strip `%` from `int_rate` and `revol_util`, `" months"` from `term`;
- map `emp_length` (`"< 1 year"` → 0, `"10+ years"` → 10);
- drop exact duplicate rows.

**Outlier clipping uses fixed domain bounds, never percentiles.** A percentile bound is a
statistic computed from the data — using one here would let the test period's income
distribution influence how training rows are transformed. `annual_inc` is capped at
\$1.5 M and `dti` at 60 because those are the limits of plausible underwriting, not
because of where any quantile happens to fall.

In [ ]:
# --- Strip the messy string encodings ---------------------------------------------
for col in ("int_rate", "revol_util"):
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = pd.to_numeric(df[col].str.rstrip("%").str.strip(), errors="coerce")

if not pd.api.types.is_numeric_dtype(df["term"]):
    df["term"] = pd.to_numeric(df["term"].str.replace(" months", "", regex=False).str.strip(),
                               errors="coerce")

EMP_LENGTH_MAP = {"< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
                  "5 years": 5, "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9,
                  "10+ years": 10}
if not pd.api.types.is_numeric_dtype(df["emp_length"]):
    df["emp_length"] = df["emp_length"].map(EMP_LENGTH_MAP)

# --- Fixed domain bounds (NOT percentiles) -----------------------------------------
DOMAIN_BOUNDS = {
    "annual_inc": (0, 1_500_000),
    "dti": (0, 60),
    "revol_util": (0, 150),
    "int_rate": (0, 35),
}
for col, (lo, hi) in DOMAIN_BOUNDS.items():
    n_clipped = int(((df[col] < lo) | (df[col] > hi)).sum())
    df[col] = df[col].clip(lo, hi)
    if n_clipped:
        print(f"clipped {n_clipped:>7,} values in {col} to [{lo}, {hi}]")

# --- Duplicates ---------------------------------------------------------------------
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nDropped {n_before - len(df):,} duplicate rows -> {len(df):,} remain")

### 6.3 Why is it missing? — the missingness taxonomy

The process insists on distinguishing **"missing because it does not exist"** from
**"missing because it was not recorded"**. The distinction is not academic: in the first
case the blank *is the information*, and imputing a median destroys it.

| Column | Why it is blank | Type | Treatment |
|---|---|---|---|
| `mths_since_last_delinq` | The applicant has **never been delinquent** | **Structural** — the event never happened | Add a missing **indicator**, then impute. The indicator is likely protective |
| `emp_length` | Not supplied on the form; unemployed, retired or self-employed | Mixed | Indicator + impute — "did not state" may itself carry signal |
| `revol_util` | No revolving accounts, so utilisation is undefined | **Structural** | Indicator + impute |
| `dti` | Not computable (e.g. zero reported income) | Not recorded | Impute only |
| `mort_acc`, `pub_rec_bankruptcies` | Bureau field not returned for older vintages | **Not recorded** — a reporting-era artefact | Impute only |

The important line is the first. `mths_since_last_delinq` is blank for roughly half of all
applicants, and **that blank means "clean record"** — the single most protective fact in
the file. Median-imputing it says instead "this person's last delinquency was a typical
number of months ago", which is precisely backwards.

So for structurally-missing columns we add an explicit `_was_missing` indicator **before**
imputation. The imputer in notebook 03 then fills the value, while the indicator preserves
the fact that it was absent — and the model can learn what absence means.

In [ ]:
# Columns whose blankness is itself information (see the table above).
STRUCTURALLY_MISSING = ["mths_since_last_delinq", "revol_util", "emp_length"]

indicators_added = []
for col in STRUCTURALLY_MISSING:
    if col in df.columns and df[col].isna().any():
        df[f"{col}_was_missing"] = df[col].isna().astype(int)
        indicators_added.append(f"{col}_was_missing")

if indicators_added:
    train_rows_tmp = df.loc[df["is_train"]]
    summary = pd.DataFrame([{
        "indicator": ind,
        "share missing (train)": train_rows_tmp[ind].mean(),
        "default rate WHEN missing": train_rows_tmp.loc[train_rows_tmp[ind] == 1, "target"].mean(),
        "default rate when present": train_rows_tmp.loc[train_rows_tmp[ind] == 0, "target"].mean(),
    } for ind in indicators_added]).set_index("indicator")
    display(summary.style.format("{:.3f}").set_caption(
        "Table 6 - Missing indicators: is absence itself predictive? (training rows only)"))
else:
    print("No structurally-missing columns present in this run.")

**Table 6 — Do the missing indicators carry signal?**

Compare the last two columns. Where the default rate differs between "value missing" and
"value present", the blank is genuinely informative and the indicator earns its place as a
feature. For `mths_since_last_delinq` expect the missing group — applicants with **no**
delinquency history — to default *less*, confirming that the blank means "clean record"
rather than "unknown".

Note this table is computed on **training rows only**, like everything else in Step 5/6
that involves the target.

**Action →** Keep the `_was_missing` indicators as model features in notebook 03. If a
future column shows no difference between the two rates, drop its indicator — it is adding
a column without adding information.

### 6.4 Column screening — from the training rows only

A missing rate is a **statistic**. Measuring it on the full frame would let the test
period influence which columns the model is allowed to use — a small but real form of
train–test contamination.

So the threshold is measured on **pre-cutoff rows only**, and the resulting drop list is
then applied to both sides. Same for near-constant columns.

In [ ]:
MISSING_THRESHOLD = 0.60      # drop a column missing in >60% of TRAINING rows
NUNIQUE_THRESHOLD = 1         # drop constant columns

train_rows = df.loc[df["is_train"]]
protected = {"target", "is_train", "is_train_random_ref", "issue_d", "earliest_cr_line"}

miss_rate = train_rows.drop(columns=list(protected & set(df.columns))).isna().mean()
too_sparse = miss_rate[miss_rate > MISSING_THRESHOLD].index.tolist()
constant = [c for c in train_rows.columns
            if c not in protected and train_rows[c].nunique(dropna=False) <= NUNIQUE_THRESHOLD]

to_drop = sorted(set(too_sparse) | set(constant))
if to_drop:
    print("Dropped (train-measured):")
    for c in to_drop:
        print(f"  {c:<28} train missing {miss_rate.get(c, 0):.1%}")
    df = df.drop(columns=to_drop)
else:
    print("No columns dropped by the train-only screen.")

print(f"\nRemaining missingness (training rows, non-zero only):")
print((miss_rate[miss_rate > 0].drop(labels=to_drop, errors="ignore")
       .sort_values(ascending=False).map("{:.2%}".format).to_string() or "  none"))

## Step 7 — Feature engineering

All row-wise, so all safe on the full frame. The engineered features encode the ratios an
underwriter actually reasons about, which a raw column pair does not give a linear model
for free:

| Feature | Definition | Why |
|---|---|---|
| `credit_history_years` | `issue_d − earliest_cr_line` | Length of credit file — thin files are riskier |
| `fico_avg` | midpoint of the FICO band | The band is reported as a range; the midpoint is the usable scalar |
| `loan_to_income` | `loan_amnt / annual_inc` | Leverage — the single most interpretable affordability ratio |
| `installment_to_income` | annualised `installment / annual_inc` | Payment burden, which `dti` omits for *this* loan |
| `revol_util_bucket` | fixed edges 0/25/50/75/100 | Utilisation has known regulatory-style break points |
| `is_36_month` | `term == 36` | 60-month loans are structurally riskier |
| `grade_ordinal`, `sub_grade_ordinal` | A→0 … G→6 | Grades are **ordered**; one-hot would discard that |

`credit_history_years` uses `issue_d` as a proxy for the application date. That is
standard practice on this dataset, but the dataset never records the true decision
moment — see the limitations in notebook 03.

`revol_util_bucket` uses **fixed** edges. Quantile edges would be learned from the data
and would leak the test distribution.

### Why target encoding is deliberately NOT used

The obvious next feature would be **target encoding** — replacing `addr_state` or
`purpose` with the mean default rate of that group. It is powerful, and it is the single
most dangerous transform in this project's vocabulary.

The problem: computed on the rows the model trains on, every row has partly seen **its
own answer**. A state with one loan gets an encoding equal to that loan's outcome. The
model then scores brilliantly in validation and collapses in production.

It is not used here. If it ever is, the only safe form is
`sklearn.preprocessing.TargetEncoder` **inside the `Pipeline`**, which fits out-of-fold —
so each row is encoded using data that excludes it, and each CV fold refits the encoder on
that fold's training portion only:

```python
from sklearn.preprocessing import TargetEncoder     # scikit-learn >= 1.3

Pipeline([
    ("te", TargetEncoder(cv=5)),   # out-of-fold by construction
    ("clf", HistGradientBoostingClassifier()),
])
```

Computing it once over the whole frame — the intuitive way — is the textbook indirect
leakage the process appendix warns about.

In [ ]:
# Ordinal encodings: fixed, domain-defined mappings -- nothing learned from data.
GRADE_ORDER = list("ABCDEFG")
df["grade_ordinal"] = df["grade"].map({g: i for i, g in enumerate(GRADE_ORDER)})
df["sub_grade_ordinal"] = (
    df["sub_grade"].str[0].map({g: i for i, g in enumerate(GRADE_ORDER)}) * 5
    + pd.to_numeric(df["sub_grade"].str[1], errors="coerce") - 1
)

# Row-wise derived features.
df["credit_history_years"] = (
    (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25).clip(0, 70)
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
df["loan_to_income"] = (df["loan_amnt"] / df["annual_inc"].replace(0, np.nan)).clip(0, 5)
df["installment_to_income"] = (
    df["installment"] * 12 / df["annual_inc"].replace(0, np.nan)).clip(0, 5)
df["is_36_month"] = (df["term"] == 36).astype(int)

# Fixed bucket edges -- NOT quantiles.
REVOL_UTIL_EDGES = [-0.01, 25, 50, 75, 100, 150]
REVOL_UTIL_LABELS = ["0-25", "25-50", "50-75", "75-100", "100+"]
df["revol_util_bucket"] = pd.cut(df["revol_util"], bins=REVOL_UTIL_EDGES,
                                 labels=REVOL_UTIL_LABELS)

# issue_d and earliest_cr_line are kept for the split/CV ordering, not as model features.
engineered = ["grade_ordinal", "sub_grade_ordinal", "credit_history_years", "fico_avg",
              "loan_to_income", "installment_to_income", "is_36_month", "revol_util_bucket"]
print(f"Added {len(engineered)} engineered features")
df[engineered].describe(include="all").T

## Step 5 — Exploratory data analysis (training rows only)

Every figure below is drawn from **pre-cutoff rows**. This is not pedantry: choosing
features or model families after eyeballing the test period is a real, if soft, form of
leakage, and it is the one nobody notices. The test period stays unseen until notebook 03
reports on it.

Two figures in section 12 deliberately span the full date range — they are about the
*split protocol itself*, not about model selection.

In [ ]:
eda = df.loc[df["is_train"]].copy()
print(f"EDA set: {len(eda):,} pre-cutoff loans "
      f"({eda['issue_d'].min():%Y-%m} to {eda['issue_d'].max():%Y-%m})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = eda["target"].value_counts().sort_index()
bars = ax.bar([OUTCOME_LABELS[i] for i in counts.index], counts.values,
              color=[OUTCOME_COLORS[i] for i in counts.index], width=0.55)
for bar, value in zip(bars, counts.values):
    ax.annotate(f"{value:,}\n({value / counts.sum():.1%})",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=10, color="#52514e")
ax.set_ylabel("Loans")
ax.set_title("Class balance, training period")
ax.set_ylim(0, counts.max() * 1.18)
save_fig("01_class_balance")

**Figure 1 — Class balance in the training period.**

Roughly one loan in five is charged off. That imbalance drives three decisions later:
ROC-AUC alone will look flattering because true negatives dominate, so PR-AUC is reported
alongside it; the classifiers use `class_weight="balanced"`; and the decision threshold is
chosen explicitly rather than left at 0.5, which is meaningless at this prevalence.

The imbalance is real but moderate — this is not a rare-event problem, so resampling
techniques such as SMOTE are not warranted.

**Action →** Use `class_weight="balanced"` rather than resampling in notebook 03, report
PR-AUC against the prevalence floor set in notebook 01, and derive the threshold from the
cost matrix instead of defaulting to 0.5.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
by_grade = (eda.groupby("grade", observed=True)
              .agg(default_rate=("target", "mean"), n=("target", "size"))
              .reindex(GRADE_ORDER).dropna())
bars = ax.bar(by_grade.index, by_grade["default_rate"], color=C_BLUE, width=0.6)
for bar, (rate, n) in zip(bars, by_grade[["default_rate", "n"]].values):
    ax.annotate(f"{rate:.1%}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=9.5, color="#52514e")
ax.axhline(eda["target"].mean(), color=C_GRAY, ls="--", lw=1.5,
           label=f"overall {eda['target'].mean():.1%}")
ax.set_ylabel("Default rate")
ax.set_xlabel("Grade assigned at origination")
ax.set_title("Default rate by Lending Club grade")
ax.set_ylim(0, by_grade["default_rate"].max() * 1.25)
ax.legend(frameon=False)
save_fig("02_default_rate_by_grade")

**Figure 2 — Default rate by grade.**

The rate rises monotonically from A to G, which tells us two things. First, Lending Club's
own grade is a strong, well-ordered predictor — any model we build has to beat it, not
merely correlate with it. Second, because the ordering is genuine, `grade` is encoded
**ordinally**; one-hot encoding would throw away exactly the structure this figure shows.

The monotonicity is also a useful sanity check on the data pipeline: a non-monotonic
pattern here would suggest the target or the grade column had been mangled upstream.

**Action →** Encode `grade`/`sub_grade` **ordinally**, not one-hot (done in Step 7), and
use `sub_grade` as baseline B1 in notebook 03 Step 8 — this figure is the evidence that
the incumbent is a serious bar to clear.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
by_purpose = (eda.groupby("purpose", observed=True)
                .agg(default_rate=("target", "mean"), n=("target", "size"))
                .query("n >= 100").sort_values("default_rate"))
ax.barh(by_purpose.index, by_purpose["default_rate"], color=C_BLUE, height=0.65)
for y, (rate, n) in enumerate(by_purpose[["default_rate", "n"]].values):
    ax.annotate(f"{rate:.1%}  (n={int(n):,})", (rate, y), xytext=(4, 0),
                textcoords="offset points", va="center", fontsize=9, color="#52514e")
ax.axvline(eda["target"].mean(), color=C_GRAY, ls="--", lw=1.5)
ax.set_xlabel("Default rate")
ax.set_title("Default rate by stated loan purpose (>= 100 loans)")
ax.set_xlim(0, by_purpose["default_rate"].max() * 1.35)
save_fig("03_default_rate_by_purpose")

**Figure 3 — Default rate by stated purpose.**

`small_business` sits well above the portfolio average while `credit_card` and `car` sit
below it. The spread justifies keeping `purpose` as a feature, but two cautions apply.

The category is **self-reported and unverified**, so it measures what the applicant said
as much as what the money did. And the low-volume categories carry wide sampling error —
the counts are printed next to each bar for exactly that reason. The `min_frequency`
setting on the one-hot encoder in notebook 03 folds the thinnest categories together
rather than letting them contribute noisy, high-variance dummies.

**Action →** Keep `purpose` as a feature with `min_frequency=0.01` on the encoder, and
slice the Step 9 error analysis by `purpose` to check the model is not systematically
wrong on the low-volume categories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for outcome in (0, 1):
    subset = eda.loc[eda["target"] == outcome, "int_rate"].dropna()
    axes[0].hist(subset, bins=45, density=True, alpha=0.55,
                 color=OUTCOME_COLORS[outcome], label=OUTCOME_LABELS[outcome])
axes[0].set_xlabel("Interest rate (%)")
axes[0].set_ylabel("Density")
axes[0].set_title("Interest rate by outcome")
axes[0].legend(frameon=False)

fico_bins = pd.cut(eda["fico_avg"], bins=np.arange(660, 850, 15))
fico_curve = eda.groupby(fico_bins, observed=True)["target"].agg(["mean", "size"])
fico_curve = fico_curve[fico_curve["size"] >= 50]
centres = [interval.mid for interval in fico_curve.index]
axes[1].plot(centres, fico_curve["mean"], color=C_BLUE, lw=2, marker="o", ms=6)
axes[1].axhline(eda["target"].mean(), color=C_GRAY, ls="--", lw=1.5,
                label=f"overall {eda['target'].mean():.1%}")
axes[1].set_xlabel("FICO score (band midpoint)")
axes[1].set_ylabel("Default rate")
axes[1].set_title("Default rate by FICO band")
axes[1].legend(frameon=False)

plt.tight_layout()
save_fig("04_int_rate_and_fico")

**Figure 4 — Interest rate by outcome (left) and default rate by FICO band (right).**

The two panels say the same thing from opposite ends. Charged-off loans are priced
visibly higher: the distribution is shifted right, confirming that Lending Club's pricing
already encodes real risk information. And default rate falls steadily as FICO rises.

The overlap in the left panel matters more than the separation. Interest rate is
informative but nowhere near separating — which is precisely why an AUC around 0.70 is
the honest expectation for this problem, and why a model reporting 0.95 should be
suspected of leakage rather than celebrated.

A modelling consequence: `int_rate` is Lending Club's own risk assessment, not raw
applicant data. Keeping it means partly predicting their model. It is retained here
because it is genuinely available at application time, but its dominance in the feature
importance plot in notebook 03 should be read in that light.

**Action →** Report feature importance in Step 10 with this caveat attached, and treat any
result where `int_rate`/`sub_grade` dominate as evidence the model is re-learning the
incumbent rather than adding information.

In [ ]:
num_cols = ["loan_amnt", "int_rate", "installment", "annual_inc", "dti", "fico_avg",
            "revol_util", "revol_bal", "open_acc", "total_acc", "credit_history_years",
            "loan_to_income", "installment_to_income", "grade_ordinal", "target"]
num_cols = [c for c in num_cols if c in eda.columns]
corr = eda[num_cols].corr()

# Diverging blue<->red with a neutral gray midpoint (never a rainbow).
from matplotlib.colors import LinearSegmentedColormap
diverging = LinearSegmentedColormap.from_list("blue_gray_red", [C_BLUE, "#f0efec", C_RED])

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, cmap=diverging, vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.5, linecolor="#fcfcfb", annot=True, fmt=".2f",
            annot_kws={"size": 7}, cbar_kws={"shrink": 0.7, "label": "Pearson r"}, ax=ax)
ax.set_title("Correlation between numeric features (training period)")
save_fig("05_correlation_heatmap")

**Figure 5 — Correlation matrix.**

Three things to take from it.

`loan_amnt` and `installment` are almost perfectly correlated — unavoidable, since the
instalment is a deterministic function of amount, rate and term. Collinearity of this kind
inflates the variance of logistic-regression coefficients and makes them unstable to
interpret, though it does not hurt the tree ensembles. This is one reason the report leans
on permutation importance rather than raw coefficients when ranking features.

`int_rate` and `grade_ordinal` are likewise near-duplicates, as expected: the grade *sets*
the price.

No individual feature correlates strongly with `target`. That is normal and expected here
— the signal in credit scoring is diffuse and additive across many weak predictors, which
is why the ensembles are worth fitting at all.

**Action →** Prefer permutation importance and SHAP over raw logistic coefficients when
ranking features in Step 10, since the `loan_amnt`/`installment` collinearity makes the
coefficients unstable to interpret.

In [ ]:
missing = eda.isna().mean()
missing = missing[missing > 0].sort_values()
if len(missing):
    fig, ax = plt.subplots(figsize=(7.5, max(2.5, 0.34 * len(missing))))
    ax.barh(missing.index, missing.values, color=C_BLUE, height=0.6)
    for y, value in enumerate(missing.values):
        ax.annotate(f"{value:.1%}", (value, y), xytext=(4, 0), textcoords="offset points",
                    va="center", fontsize=9, color="#52514e")
    ax.set_xlabel("Share of training rows missing")
    ax.set_title("Missingness by column (training period)")
    ax.set_xlim(0, min(1.0, missing.max() * 1.3))
    save_fig("06_missingness")
else:
    print("No missing values remain in the training rows.")

**Figure 6 — Missingness in the training period.**

Whatever survived the 60 % screen in section 9 is imputed inside the modelling pipeline —
median for numeric columns, most-frequent for categoricals — and, critically, **the
imputation values are learned from training rows only**. Computing a median over the full
frame is the textbook contamination mistake, and it would be invisible in any metric.

Note that missingness here is not always random. `mths_since_last_delinq` is missing
precisely for applicants who have never been delinquent, so "missing" carries signal. The
tree-based models can exploit that directly; median imputation partly obscures it. Adding
explicit missing-indicator columns is the natural next iteration.

**Action →** Already acted on — the `_was_missing` indicators added in Step 6.3 preserve
exactly this information for the model.

### 5.8 Drift — the evidence for an out-of-time split

Unlike section 11, these two figures span the **full** date range with the cutoff marked.
They describe the split protocol itself rather than informing feature or model choice, so
showing the post-cutoff period here does not contaminate model selection.

In [ ]:
monthly = (df.set_index("issue_d")
             .groupby(pd.Grouper(freq="MS"))
             .agg(volume=("target", "size"), default_rate=("target", "mean"))
             .query("volume > 0"))

fig, axes = plt.subplots(2, 1, figsize=(9.5, 6.5), sharex=True)

axes[0].fill_between(monthly.index, monthly["volume"], color=C_BLUE, alpha=0.25)
axes[0].plot(monthly.index, monthly["volume"], color=C_BLUE, lw=2)
axes[0].set_ylabel("Loans issued")
axes[0].set_title("Origination volume and default rate over time")

axes[1].plot(monthly.index, monthly["default_rate"], color=C_ORANGE, lw=2)
axes[1].set_ylabel("Default rate")
axes[1].set_xlabel("Issue month")

for ax in axes:
    ax.axvline(CUTOFF_DATE, color=C_GRAY, ls="--", lw=1.8)
    ax.annotate("train  |  test", (CUTOFF_DATE, ax.get_ylim()[1]),
                xytext=(0, -14), textcoords="offset points",
                ha="center", fontsize=9.5, color="#52514e",
                bbox=dict(fc="#fcfcfb", ec="none", alpha=0.85))
plt.tight_layout()
save_fig("07_drift_over_time")

**Figure 7 — Origination volume (top) and realised default rate (bottom) by issue
month, with the train/test cutoff marked.**

This is the empirical case for the out-of-time protocol. Volume grows by orders of
magnitude across the window, so a random split would draw its test set overwhelmingly from
the *late*, high-volume years while still letting the model train on them — the exact
look-ahead an OOT split forbids. The default rate is visibly non-stationary too, so the
relationship being learned is not fixed across the window.

The bottom panel also shows the right-censoring artefact directly: the realised default
rate turns sharply toward the right-hand edge. Loans issued in the last months of the
window have only had time to resolve if they resolved *early*, and early resolution is
disproportionately charge-off. This is why the test-period default rate is not the true
one, and it is recorded as limitation 2 in notebook 03.

**Action →** Keep the out-of-time protocol, and carry the censoring caveat into the
limitations in notebook 03 — the test-period default rate must never be quoted as the
true cohort rate.

## Export for notebook 03

### Record the split manifest

Written now that every row-removal step is complete, so the counts describe exactly the
frame being exported. `test_index_md5` lets any other group confirm they reproduced the
identical split.

In [ ]:
# Auditable, shareable record of the split, computed on the FINAL row set.
train_mask, test_mask = df["is_train"], ~df["is_train"]
test_positions = np.flatnonzero(test_mask.to_numpy())
manifest = {
    "split_type": "out_of_time",
    "cutoff_date": str(CUTOFF_DATE.date()),
    "cutoff_rule": "issue_d < cutoff -> train; issue_d >= cutoff -> test",
    "data_source": DATA_SOURCE,
    "random_seed": RANDOM_SEED,
    "n_rows_total": int(len(df)),
    "n_train": int(train_mask.sum()),
    "n_test": int(test_mask.sum()),
    "default_rate_train": float(df.loc[train_mask, "target"].mean()),
    "default_rate_test": float(df.loc[test_mask, "target"].mean()),
    "train_date_range": [str(df.loc[train_mask, "issue_d"].min().date()),
                         str(df.loc[train_mask, "issue_d"].max().date())],
    "test_date_range": [str(df.loc[test_mask, "issue_d"].min().date()),
                        str(df.loc[test_mask, "issue_d"].max().date())],
    "test_index_md5": hashlib.md5(test_positions.tobytes()).hexdigest(),
    "reference_split": {
        "note": "Random stratified split, reported for comparison only. Not the protocol.",
        "test_size": 0.2,
        "random_state": RANDOM_SEED,
    },
}
(ARTIFACTS / "split_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

In [ ]:
# Deliberately NOT applied here: imputation, scaling, encoding. Those learn parameters
# from data and belong inside the Pipeline in notebook 03, fitted on training rows only.
OUT_PATH = DATA_PROCESSED / "clean.parquet"
df.to_parquet(OUT_PATH, index=False)

print(f"Wrote {OUT_PATH}")
print(f"  rows      : {len(df):,}  (train {df['is_train'].sum():,} / "
      f"test {(~df['is_train']).sum():,})")
print(f"  columns   : {df.shape[1]}")
print(f"  source    : {DATA_SOURCE}")
print(f"  cutoff    : {CUTOFF_DATE.date()}")
print("\nNext: run notebooks/03_modeling_and_evaluation.ipynb")
df.dtypes.to_frame("dtype")